# Retrieve Supabase Templates

Experiment to retrieve rows from the `templates` table in Supabase.

In [1]:
from dotenv import load_dotenv
from supabase import create_client
import os
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[1]
load_dotenv(dotenv_path=PROJECT_ROOT / ".env")

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError("SUPABASE_URL and SUPABASE_KEY must be defined in the .env file.")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print("Connected to Supabase")

Connected to Supabase


In [2]:
response = supabase.table("templates").select("*").execute()
templates = response.data
print(f"Retrieved {len(templates)} templates.")

Retrieved 5 templates.


In [3]:
if templates:
    import json
    print(json.dumps(templates[0], indent=2))

{
  "id": "zettelkasten",
  "name": "Zettelkasten",
  "description": "Atomic note format emphasizing one idea per note and explicit links between concepts.",
  "instructions": "Capture a single idea per note. Use concise language and create links to related concepts whenever possible.",
  "category": "Knowledge Management",
  "difficulty": null,
  "tags": [
    "notes",
    "knowledge-management",
    "atomic-notes",
    "research"
  ],
  "version": 1,
  "preview_markdown": "---\nid: 202606210201\ntitle: \"Quantum Entanglement\"\ntags: [physics, quantum-mechanics]\naliases: [Spooky Action]\n---\n\n# Quantum Entanglement\n\nQuantum entanglement describes the phenomenon where two particles become correlated in such a way that measuring one affects the state of the other.\n\n## Related Concepts\n\n- [[Wave Function Collapse]]\n- [[Quantum Computing]]\n",
  "template_markdown": "---\nid: [NOTE_ID]\ntitle: \"[TITLE]\"\ntags: [tag1, tag2]\naliases: []\n---\n\n# [TITLE]\n\nWrite a concise des

In [4]:
from IPython.display import display, Markdown

# Render a simple Markdown string
display(Markdown(templates[0]['template_markdown']))


---
id: [NOTE_ID]
title: "[TITLE]"
tags: [tag1, tag2]
aliases: []
---

# [TITLE]

Write a concise description of a single concept.

## Related Concepts

- [[Related Note 1]]
- [[Related Note 2]]


In [5]:
from IPython.display import display, Markdown

# Render a simple Markdown string
display(Markdown(templates[1]['preview_markdown']))


# Topic: TCP Handshake

## 1. Plain Explanation

> Explain it as if speaking to a child. Avoid jargon.

Before two computers can talk, they first greet each other and agree that both are ready to communicate.

## 2. Identified Knowledge Gaps

- [ ] Unsure why the ACK packet is required.
- [ ] Need to review retransmissions.

## 3. Refined Source Material

- [RFC 793](https://datatracker.ietf.org/doc/html/rfc793) — Defines TCP behavior.


--- 
---


# Learner & Parser Agent Drafts

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from agents.note_profiler.app.agent import root_agent
from google.adk.apps import App
from google.adk.runners import InMemoryRunner
from google.genai import types

app = App(name="app", root_agent=root_agent)
runner = InMemoryRunner(app=app)

session = await runner.session_service.create_session(
    app_name="app", user_id="dev_user"
)
refactored_templates = []


# Draft pipeline execution to Note Profiler Agent:
for note in templates[:]:
    agent_input = f"Name: {note.get('name', '')}\nInstructions: {note.get('instructions', '')}\nTemplate: {note.get('template_markdown', '')}"
    
    print(f"Running Note Profiler for {note.get('name')}...\n")
    enriched_profile = None
    
    async for event in runner.run_async(
        user_id="dev_user",
        session_id=session.id,
        new_message=types.Content(role="user", parts=[types.Part.from_text(text=agent_input)]),
    ):
        # 1. Check for the structured Pydantic output
        if event.actions.state_delta.get('profile_data') is not None:
            enriched_profile = session.state['profile_data'] = event.actions.state_delta['profile_data']
            
        # 2. Print the raw text/errors from the LLM to diagnose why output is None
        if event.content and event.content.parts:
            print(f"RAW EVENT CONTENT: {event.content.parts[0].text[:500]}...")
            
    print("\n" + "="*40)
    print(f"Final State profile_data: {session.state.get('profile_data') is not None}")
    print(f"Final Enriched Profile: {type(enriched_profile)}")
    print("="*40 + "\n")
    
    # if enriched_profile:
    #     print(enriched_profile.model_dump_json(indent=2))
    # else:
    #     print("No structured output was generated. Please review the RAW EVENT CONTENT above.")

refactored_templates.append(enriched_profile)

Running Note Profiler for Zettelkasten...

RAW EVENT CONTENT: {"id":"zettelkasten","name":"Zettelkasten","description":"A template designed for capturing single ideas or concepts in a concise manner, promoting interlinking and easy reference.","instructions":"Capture a single idea per note. Use concise language and create links to related concepts whenever possible.","preview":"---\nid: [NOTE_ID]\ntitle: \"[TITLE]\"\ntags: [tag1, tag2]\naliases: []\n---\n\n# [TITLE]\n\nWrite a concise description of a single concept.\n\n## Related Concepts\n\n- [[Related N...

Final State profile_data: False
Final Enriched Profile: <class 'NoneType'>

No structured output was generated. Please review the RAW EVENT CONTENT above.


In [52]:
event.state

AttributeError: 'Event' object has no attribute 'state'

In [26]:
event.content.parts[0].text

'{"id":"zettelkasten","name":"Zettelkasten","description":"A template designed for capturing single ideas or concepts in a concise manner, promoting interlinking and easy reference.","instructions":"Capture a single idea per note. Use concise language and create links to related concepts whenever possible.","preview":"---\\nid: [NOTE_ID]\\ntitle: \\"[TITLE]\\"\\ntags: [tag1, tag2]\\naliases: []\\n---\\n\\n# [TITLE]\\n\\nWrite a concise description of a single concept.\\n\\n## Related Concepts\\n\\n- [[Related Note 1]]\\n- [[Related Note 2]]","tags":["knowledge capture","personal knowledge management","idea capture","note taking","research","literature notes"],"purpose":["knowledge capture","idea capture","note taking","literature notes","reference collection"],"sections":["title","description","related concepts"],"organization_structure":["bullet lists","linked notes","conceptual links"],"style":["concise","informal","exploratory","reflective","structured"],"embedding_text":"This templ

In [ ]:
session.state = event.actions.state_delta


In [ ]:
(event.actions.state_delta)

AttributeError: 'dict' object has no attribute 'model_dump_json'

In [45]:
event.actions.__dict__

{'skip_summarization': None,
 'state_delta': {'profile_data': {'id': 'zettelkasten',
   'name': 'Zettelkasten',
   'description': 'A template designed for capturing single ideas or concepts in a concise manner, promoting interlinking and easy reference.',
   'instructions': 'Capture a single idea per note. Use concise language and create links to related concepts whenever possible.',
   'preview': '---\nid: [NOTE_ID]\ntitle: "[TITLE]"\ntags: [tag1, tag2]\naliases: []\n---\n\n# [TITLE]\n\nWrite a concise description of a single concept.\n\n## Related Concepts\n\n- [[Related Note 1]]\n- [[Related Note 2]]',
   'tags': ['knowledge capture',
    'personal knowledge management',
    'idea capture',
    'note taking',
    'research',
    'literature notes'],
   'purpose': ['knowledge capture',
    'idea capture',
    'note taking',
    'literature notes',
    'reference collection'],
   'sections': ['title', 'description', 'related concepts'],
   'organization_structure': ['bullet lists',
  

In [8]:
display(Markdown(agent_input))

Name: Zettelkasten
Instructions: Capture a single idea per note. Use concise language and create links to related concepts whenever possible.
Template: ---
id: [NOTE_ID]
title: "[TITLE]"
tags: [tag1, tag2]
aliases: []
---

# [TITLE]

Write a concise description of a single concept.

## Related Concepts

- [[Related Note 1]]
- [[Related Note 2]]


In [9]:
note

{'id': 'zettelkasten',
 'name': 'Zettelkasten',
 'description': 'Atomic note format emphasizing one idea per note and explicit links between concepts.',
 'instructions': 'Capture a single idea per note. Use concise language and create links to related concepts whenever possible.',
 'category': 'Knowledge Management',
 'difficulty': None,
 'tags': ['notes', 'knowledge-management', 'atomic-notes', 'research'],
 'version': 1,
 'preview_markdown': '---\nid: 202606210201\ntitle: "Quantum Entanglement"\ntags: [physics, quantum-mechanics]\naliases: [Spooky Action]\n---\n\n# Quantum Entanglement\n\nQuantum entanglement describes the phenomenon where two particles become correlated in such a way that measuring one affects the state of the other.\n\n## Related Concepts\n\n- [[Wave Function Collapse]]\n- [[Quantum Computing]]\n',
 'template_markdown': '---\nid: [NOTE_ID]\ntitle: "[TITLE]"\ntags: [tag1, tag2]\naliases: []\n---\n\n# [TITLE]\n\nWrite a concise description of a single concept.\n\n## 

In [10]:
agent_input

'Name: Zettelkasten\nInstructions: Capture a single idea per note. Use concise language and create links to related concepts whenever possible.\nTemplate: ---\nid: [NOTE_ID]\ntitle: "[TITLE]"\ntags: [tag1, tag2]\naliases: []\n---\n\n# [TITLE]\n\nWrite a concise description of a single concept.\n\n## Related Concepts\n\n- [[Related Note 1]]\n- [[Related Note 2]]\n'

In [11]:
event.finish_reason


<FinishReason.STOP: 'STOP'>

In [12]:

event.is_final_response()

True

--- 
---


# Selector Agent Drafts